# 02.3 — BM25 (Best Match 25)

**Problem it solves:** TF-IDF has two weaknesses: (1) raw TF grows unboundedly (one extra mention boosts score forever), (2) long documents are penalized even when they're genuinely relevant.

**BM25 fixes both with two parameters:**
- **k1** (typically 1.2–2.0): controls TF saturation. High k1 = TF matters more. k1→∞ = raw TF.
- **b** (typically 0.75): controls length normalization. b=1 = full normalization. b=0 = no length normalization.

**Formula:**
```
BM25(t,d) = IDF(t) × [TF(t,d) × (k1+1)] / [TF(t,d) + k1 × (1 - b + b × |d|/avgdl)]
```

This is the backbone of Elasticsearch, Lucene, and most production search systems.

---

In [ ]:
import math
import numpy as np
from collections import Counter
import re

def tokenize(text: str) -> list[str]:
    return re.findall(r'\b[a-z]+\b', text.lower())

class BM25:
    """
    BM25 (Okapi BM25) scorer.
    Robertson & Spärck Jones, 1994.
    """
    
    def __init__(self, k1: float = 1.5, b: float = 0.75):
        """
        k1: term frequency saturation parameter
            k1=0: pure IDF (TF ignored)
            k1=1.2-2.0: typical range
            k1→inf: approaches TF-IDF behavior
        b: length normalization parameter  
            b=0: no length normalization
            b=1: full normalization (scores independent of doc length)
            b=0.75: typical value
        """
        self.k1 = k1
        self.b = b
        self.corpus_size = 0
        self.avgdl = 0
        self.doc_freqs: list[Counter] = []   # tf per doc
        self.doc_lens: list[int] = []
        self.df: Counter = Counter()          # document frequency per term
        self.idf: dict[str, float] = {}
        self.docs: list[str] = []
    
    def fit(self, docs: list[str]):
        self.docs = docs
        self.corpus_size = len(docs)
        
        # Tokenize and compute TF per doc
        for doc in docs:
            tokens = tokenize(doc)
            self.doc_freqs.append(Counter(tokens))
            self.doc_lens.append(len(tokens))
            # Update document frequency
            self.df.update(set(tokens))
        
        self.avgdl = sum(self.doc_lens) / self.corpus_size
        
        # BM25 IDF: log((N - df + 0.5) / (df + 0.5) + 1)
        # The +1 ensures IDF is always positive
        # This is the Robertson-Spärck Jones IDF variant
        N = self.corpus_size
        self.idf = {
            term: math.log((N - freq + 0.5) / (freq + 0.5) + 1)
            for term, freq in self.df.items()
        }
        return self
    
    def score(self, query: str, doc_id: int) -> float:
        """Score a single query-document pair."""
        query_terms = tokenize(query)
        doc_tf = self.doc_freqs[doc_id]
        doc_len = self.doc_lens[doc_id]
        
        score = 0.0
        for term in query_terms:
            if term not in self.idf:
                continue
            
            tf = doc_tf.get(term, 0)
            idf = self.idf[term]
            
            # Length-normalized TF
            # The denominator saturates as tf grows (TF saturation)
            # The (1 - b + b * dl/avgdl) term penalizes long docs
            normalized_tf = (tf * (self.k1 + 1)) / (
                tf + self.k1 * (1 - self.b + self.b * doc_len / self.avgdl)
            )
            
            score += idf * normalized_tf
        
        return score
    
    def search(self, query: str, top_k: int = 5) -> list[tuple]:
        scores = [(self.score(query, i), i) for i in range(self.corpus_size)]
        scores.sort(reverse=True)
        return [(score, doc_id, self.docs[doc_id]) 
                for score, doc_id in scores[:top_k] if score > 0]

docs = [
    "machine learning is a subset of artificial intelligence",
    "deep learning uses neural networks with many layers",
    "natural language processing analyzes human text and speech",
    "machine translation converts text from one language to another",
    "neural networks are used in image recognition and NLP tasks",
    "information retrieval finds relevant documents for queries",
    "BM25 is a ranking function used in information retrieval",
    "TF-IDF weights terms by frequency and inverse document frequency",
]

bm25 = BM25(k1=1.5, b=0.75).fit(docs)
print(f"Corpus: {len(docs)} docs, avgdl={bm25.avgdl:.1f} tokens")

query = "information retrieval ranking"
print(f"\nQuery: {query!r}")
for score, doc_id, text in bm25.search(query):
    print(f"  [{doc_id}] score={score:.4f}  {text}")

In [ ]:
# Visualize what k1 and b actually control

print("=== Effect of k1: TF saturation ===")
print("How much does adding more occurrences of a term keep helping?")
print()

# Fix a doc with varying TF for term 'neural'
avgdl = 10
doc_len = 10
idf = 2.0

for k1 in [0.5, 1.2, 1.5, 2.0, float('inf')]:
    k1_display = f'k1={k1}' if k1 != float('inf') else 'k1=inf (TF-IDF)'
    scores = []
    for tf in [1, 2, 5, 10, 20, 50]:
        if k1 == float('inf'):
            score = idf * tf  # raw TF
        else:
            normalized_tf = tf * (k1 + 1) / (tf + k1)
            score = idf * normalized_tf
        scores.append(f'{score:.2f}')
    print(f"  {k1_display:20}  TF=[1,2,5,10,20,50] -> scores={scores}")

print()
print("=== Effect of b: length normalization ===")
print("How much does doc length affect scoring?")
print()

k1 = 1.5
tf = 3  # term appears 3 times
for b in [0.0, 0.25, 0.5, 0.75, 1.0]:
    scores = []
    for doc_len in [5, 10, 20, 50, 100]:
        norm_tf = tf * (k1+1) / (tf + k1*(1 - b + b * doc_len / avgdl))
        scores.append(f'{idf * norm_tf:.2f}')
    print(f"  b={b:.2f}   doc_len=[5,10,20,50,100] -> scores={scores}")

In [ ]:
# BM25 vs TF-IDF: the document length problem

from collections import Counter

# Two docs: same content, one has lots of filler
short_doc = "neural network classification works well"
long_doc  = "neural network classification works well " + "and so " * 50

test_docs = [short_doc, long_doc]
query = "neural network"

# BM25 scoring
bm25_test = BM25(k1=1.5, b=0.75).fit(test_docs)
bm25_short = bm25_test.score(query, 0)
bm25_long  = bm25_test.score(query, 1)

print(f"BM25 scores for query {query!r}:")
print(f"  Short doc ({len(tokenize(short_doc))} tokens): {bm25_short:.4f}")
print(f"  Long doc  ({len(tokenize(long_doc))} tokens):  {bm25_long:.4f}")
print(f"  Difference: {abs(bm25_short - bm25_long):.4f} (BM25 is robust to length)")

print()
print("BM25 b=0 (no length norm) would score the long doc higher because it has more term occurrences.")
bm25_nolen = BM25(k1=1.5, b=0.0).fit(test_docs)
print(f"BM25 (b=0) short: {bm25_nolen.score(query,0):.4f}  long: {bm25_nolen.score(query,1):.4f}")

In [ ]:
# BM25+ variant: prevents zero scores from non-overlapping terms
# and adds a lower bound delta to normalized TF

class BM25Plus(BM25):
    def __init__(self, k1=1.5, b=0.75, delta=1.0):
        super().__init__(k1=k1, b=b)
        self.delta = delta
    
    def score(self, query: str, doc_id: int) -> float:
        query_terms = tokenize(query)
        doc_tf = self.doc_freqs[doc_id]
        doc_len = self.doc_lens[doc_id]
        
        score = 0.0
        for term in query_terms:
            if term not in self.idf:
                continue
            tf = doc_tf.get(term, 0)
            if tf == 0:
                continue
            idf = self.idf[term]
            # BM25+: add delta to normalized TF (lower bound)
            normalized_tf = self.delta + (tf * (self.k1 + 1)) / (
                tf + self.k1 * (1 - self.b + self.b * doc_len / self.avgdl)
            )
            score += idf * normalized_tf
        return score

print("BM25+ (with delta) on main corpus:")
bm25plus = BM25Plus().fit(docs)
for score, doc_id, text in bm25plus.search("information retrieval"):
    print(f"  [{doc_id}] {score:.4f}  {text}")

## Summary

**BM25 is the production baseline for lexical retrieval.** It beats TF-IDF on almost every benchmark and is trivially fast to implement.

**Key hyperparameters:**
- `k1=1.2–2.0` — start with 1.5
- `b=0.75` — reduce for short-document corpora (tweets, titles)

**What breaks BM25:**
- Vocabulary mismatch: query uses 'automobile', doc says 'car' — no match
- No semantic understanding — this is why dense retrieval (DPR) was invented
- Query/document length mismatch — very short queries on very long docs are hard
- Exact string match only — typos, multilingual, synonyms all fail

BM25 is still hard to beat on many benchmarks. In the BEIR heterogeneous retrieval benchmark (2021), BM25 outperforms several dense retrieval models on certain domains.

---
**Next:** 02.4 — Query Expansion